## Camada 2 - aprendizado few-shot (in-context learning)

Aqui está um dos achados centrais da era dos LLMs: o modelo aprende
dentro da própria conversa. Alguns exemplos rotulados no prompt
(*few-shot*) e a tarefa muda de comportamento - sem treinar nada.

A mudança em relação ao notebook anterior é pequena de propósito:
consertar um prompt fraco não é trocar de modelo, é dar o que faltou
- informação e exemplos que definam a regra.


In [1]:
# Setup encapsulado: cada notebook é auto-contido (a ordem ENTRE
# notebooks é livre; dentro dele, as células rodam de cima para baixo)
import os, pathlib, sys
sys.path.append(str(pathlib.Path.cwd().parent))

import pandas as pd
from llm import chat, usage_total

print(f"Modelo: {os.environ.get('LLM_MODEL', '?')}")

df = pd.read_csv('../data/reclamacoes.csv')
CATEGORIAS = 'entrega | cobranca | produto | atendimento | app'

# O prompt zero-shot também está em 01 e bonus_evals; se alterar,
# replique nos três para manter o placar comparável.
def prompt_zero_shot(texto: str) -> str:
    return (
        'Classifique a reclamacao em UMA das categorias:\n'
        f'{CATEGORIAS}\n\n'
        f'Reclamacao: "{texto}"\n\n'
        'Responda apenas com a categoria, em minusculas, e nada mais.'
    )


Modelo: deepseek-v4.1-flash


In [2]:
# As quatro âncoras ensinam a mesma regra com superfície diferente dos
# casos de teste: nenhuma repete o texto de 3, 15 ou 17. Escolher bons
# exemplos é o novo feature engineering: qualidade > quantidade.
# FEW_SHOT também está em bonus_evals; se alterar, replique nos dois.
FEW_SHOT = [
    ('O pedido foi extraviado e nunca chegou; o estorno prometido nao saiu e a fatura seguinte veio igual', 'entrega'),
    ('Cobraram uma taxa de servico que nao estava no carrinho e o suporte diz que e normal', 'cobranca'),
    ('O produto chegou amassado e faltando uma peca; a entrega foi feita no prazo', 'produto'),
    ('Passei por quatro setores no chat e ninguem resolveu; a ultima resposta foi aguarde', 'atendimento'),
]

def prompt_few_shot(texto: str) -> str:
    exemplos = '\n'.join(f'Reclamacao: "{t}" -> {c}' for t, c in FEW_SHOT)

    return (
        f'{exemplos}\n\n'
        'Classifique pela CAUSA RAIZ da reclamacao, em UMA das categorias:\n'
        'A causa raiz e o problema que ORIGINOU a reclamacao, mesmo que o texto cite atendimento, app ou cobranca.\n'
        f'{CATEGORIAS}\n\n'
        f'Reclamacao: "{texto}"\n\n'
        'Responda apenas com a categoria, em minusculas, e nada mais.'
    )


In [3]:
# Intervenções de palco:
# 1. rodar com os 4 exemplos e comparar com a camada 1
# 2. comentar uma âncora (ex.: a de entrega) e comparar o índice 15
INDICES = [15, 16, 17]  # 15 e 17 exercitam a regra; 16 fica sem cobertura de propósito

RESULTADO_FEW_SHOT = {}

for i in INDICES:
    linha = df.iloc[i]
    
    pred = chat(
                [
                    {
                     'role': 'user', 
                     'content': prompt_few_shot(linha['texto'])
                     }
                ], temperature=0).strip().lower()
    
    RESULTADO_FEW_SHOT[i] = pred
    
    ok = 'OK' if pred == linha['categoria'] else 'ERROU'
    
    print(f'[{ok}] {linha["texto"][:55]}...')
    print(f'      esperado: {linha["categoria"]} | veio: {pred}\n')


[OK] O atendente confirmou o reembolso do produto que nunca ...
      esperado: entrega | veio: entrega



[ERROU] O desconto do cupom do app sumiu no checkout e a correç...
      esperado: cobranca | veio: app



[OK] A segunda parcela do pedido feito pelo app saiu mais ca...
      esperado: cobranca | veio: cobranca



## Fechamento da camada 2

4 exemplos em 4 linhas mudaram o placar: as armadilhas cobertas
pela regra (15 e 17) agora acertam. A 16, de propósito sem âncora,
continua resistindo - e é o gancho para o tuning de amostragem e para a
intervenção seguinte.

É o *in-context learning* em ação: a 'inteligência' está no prompt,
não nos pesos do modelo.

E a mensagem de carreira: dados de qualidade vencem modelo
maior quase sempre - e custam muito menos.


## Bônus da camada 2 - tuning de amostragem: mexeu nos parâmetros e melhorou?

Intuição comum: 'se não ficou bom, mexe no temperature/top_p'.
A célula abaixo mede isso: 2 configurações, 3 rodadas, os índices
15/16/17 (15 e 17 cobertos pela regra; 16 de propósito sem cobertura).
O nome popular é micro-tuning; aqui é só tuning de amostragem - nada
de fine-tuning.

Expectativa: temp=0 tende a repetir a saída; temp=1.0 pode variar.
O erro sistemático da 16 deve seguir igual em qualquer config.

A leitura: parâmetros de amostragem compram variação (quando
compram), não conhecimento. Falta de regra se conserta com exemplos
no prompt.

(top_p e top_k ficam de fora desta matriz: são outros botões do
mesmo tipo e não mudam a conclusão.)


In [4]:
# A matriz do tuning: 2 configs x 3 rodadas sobre os índices 15/16/17.
# As rodadas repetidas existem para expor a VARIAÇÃO: com temp > 0,
# a mesma entrada pode voltar com respostas diferentes.
# São 18 chamadas - o suficiente para ver o efeito sem gastar a cota.
CONFIGS = [
    ('temp=0  ', dict(temperature=0)),
    ('temp=1.0', dict(temperature=1.0)),
]

RODADAS = 3
INDICES = [15, 16, 17]

for nome_cfg, cfg in CONFIGS:
    acertos = 0
    por_indice = {i: [] for i in INDICES}

    for _ in range(RODADAS):
        for i in INDICES:
            linha = df.iloc[i]
            pred = chat(
                [{'role': 'user', 'content': prompt_few_shot(linha['texto'])}],
                **cfg,
            ).strip().lower()
            por_indice[i].append(pred)
            acertos += pred == linha['categoria']

    total = RODADAS * len(INDICES)
    print(f'{nome_cfg} | placar: {acertos}/{total}')

    for i in INDICES:
        esperado = df.iloc[i]['categoria']
        marcas = ' '.join('OK' if p == esperado else 'ERROU' for p in por_indice[i])
        print(f'  {i}: {marcas} | respostas distintas: {len(set(por_indice[i]))}')

print()
print('Leitura: o placar não melhora com a config; a 16 continua errando')
print('do mesmo jeito. O conserto dela exige uma regra/exemplo que ensine')
print('a causa raiz - não um parâmetro de amostragem.')
print()
print('Tokens acumulados:', usage_total())


temp=0   | placar: 6/9
  15: OK OK OK | respostas distintas: 1
  16: ERROU ERROU ERROU | respostas distintas: 1
  17: OK OK OK | respostas distintas: 1


temp=1.0 | placar: 6/9
  15: OK OK OK | respostas distintas: 1
  16: ERROU ERROU ERROU | respostas distintas: 1
  17: OK OK OK | respostas distintas: 1

Leitura: o placar não melhora com a config; a 16 continua errando
do mesmo jeito. O conserto dela exige uma regra/exemplo que ensine
a causa raiz - não um parâmetro de amostragem.

Tokens acumulados: {'prompt_tokens': 5950, 'completion_tokens': 3253, 'total_tokens': 9203}


## Intervenção: um exemplo novo para o índice 16

O tuning de amostragem mostrou o limite: parâmetros não ensinam a regra
que falta. A intervenção é um quinto exemplo e uma regra explícita que ensinam
a causa raiz do índice 16 - cupom/desconto é cobrança, mesmo quando app
e atendimento aparecem no texto.

Rode a célula e compare com o placar anterior. O exemplo é uma hipótese
de prompt: se o seu modelo ainda errar, ajuste o texto do exemplo ou da
regra para tornar a causa raiz explícita.


In [5]:
# O índice 16 mistura app, atendimento e cobranca; a causa raiz e a
# cobranca. Um quinto exemplo ensina a regra que faltava.
EXEMPLO_NOVO = (
    'A promocao anunciada no app nao foi aplicada na fatura e o atendimento nao corrigiu a cobranca',
    'cobranca',
)

FEW_SHOT_AMPLIADO = FEW_SHOT + [EXEMPLO_NOVO]

def prompt_few_shot_ampliado(texto: str) -> str:
    exemplos = '\n'.join(f'Reclamacao: "{t}" -> {c}' for t, c in FEW_SHOT_AMPLIADO)
    
    return (
        f'{exemplos}\n\n'
        'Classifique pela CAUSA RAIZ da reclamacao, em UMA das categorias:\n'
        'A causa raiz e o problema que ORIGINOU a reclamacao, mesmo que o texto cite atendimento, app ou cobranca.\n'
        'Se o problema for desconto, cupom ou valor cobrado, classifique como cobranca, mesmo que o texto cite app ou atendimento.\n'
        f'{CATEGORIAS}\n\n'
        f'Reclamacao: "{texto}"\n\n'
        'Responda apenas com a categoria, em minusculas, e nada mais.'
    )

RESULTADO_AMPLIADO = {}

for i in [15, 4, 16, 10, 17, 1]:
    linha = df.iloc[i]
    
    pred = chat(
                [
                    {
                        'role': 'user', 
                        'content': prompt_few_shot_ampliado(linha['texto'])
                    }
                ], temperature=0).strip().lower()
    
    RESULTADO_AMPLIADO[i] = pred
    
    ok = 'OK' if pred == linha['categoria'] else 'ERROU'
    
    print(f'[{ok}] {linha["texto"][:55]}...')
    print(f'      esperado: {linha["categoria"]} | veio: {pred}\n')

print('Antes -> depois nos três índices da camada:')
for i in [15, 16, 17]:
    antes = RESULTADO_FEW_SHOT.get(i, '?')
    depois = RESULTADO_AMPLIADO.get(i, '?')
    print(f'  {i}: {antes} -> {depois}')

print('Tokens acumulados:', usage_total())


[OK] O atendente confirmou o reembolso do produto que nunca ...
      esperado: entrega | veio: entrega



[OK] O aplicativo trava na tela de pagamento e não deixa fec...
      esperado: app | veio: app



[OK] O desconto do cupom do app sumiu no checkout e a correç...
      esperado: cobranca | veio: cobranca



[OK] Meu pedido voltou pro remetente sem aviso e sem eu ter ...
      esperado: entrega | veio: entrega



[OK] A segunda parcela do pedido feito pelo app saiu mais ca...
      esperado: cobranca | veio: cobranca



[OK] Cobraram o frete duas vezes na minha fatura do cartão...
      esperado: cobranca | veio: cobranca

Antes -> depois nos três índices da camada:
  15: entrega -> entrega
  16: app -> cobranca
  17: cobranca -> cobranca
Tokens acumulados: {'prompt_tokens': 8048, 'completion_tokens': 3763, 'total_tokens': 11811}
